# 🎓 Student Performance Analysis & Prediction
**Author:** Shreya Mishra | BIT Mesra, Jaipur  
**Date:** Feb 2025 – Apr 2025  
**Tools:** Pandas, Seaborn, Matplotlib, scikit-learn

---
### Objective
Analyse student exam scores across math, reading and writing. Identify key factors affecting performance and build ML models to predict average score and grade.


In [ ]:
# Install dependencies (only needed in Colab)
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully ✅')

## 1. Load Dataset

In [ ]:
# Generating sample data — replace with:  data = pd.read_csv('students.csv')
np.random.seed(42)
n = 1000

data = pd.DataFrame({
    'gender'            : np.random.choice(['Male', 'Female'], n),
    'parental_education': np.random.choice(
        ["high school", "some college", "bachelor's degree", "master's degree"], n),
    'lunch'             : np.random.choice(['standard', 'free/reduced'], n),
    'test_prep'         : np.random.choice(['completed', 'none'], n),
    'math_score'        : np.random.randint(30, 100, n),
    'reading_score'     : np.random.randint(30, 100, n),
    'writing_score'     : np.random.randint(30, 100, n),
})
data['avg_score'] = data[['math_score','reading_score','writing_score']].mean(axis=1).round(2)
data['grade']     = pd.cut(data['avg_score'],
                            bins=[0,50,60,70,80,100],
                            labels=['F','D','C','B','A'])
data.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Shape :', data.shape)
print('\nNull values :')
print(data.isnull().sum())
print('\nData Types :')
print(data.dtypes)

In [ ]:
data[['math_score','reading_score','writing_score','avg_score']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Student Performance – EDA', fontsize=15, fontweight='bold')

sns.countplot(x='grade', data=data, palette='Blues_d', order=['A','B','C','D','F'], ax=axes[0,0])
axes[0,0].set_title('Grade Distribution')

sns.boxplot(x='gender', y='avg_score', data=data, palette='Set2', ax=axes[0,1])
axes[0,1].set_title('Avg Score by Gender')

sns.barplot(x='test_prep', y='avg_score', data=data, palette='pastel', ci=None, ax=axes[1,0])
axes[1,0].set_title('Test Prep vs Avg Score')

corr = data[['math_score','reading_score','writing_score','avg_score']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,1])
axes[1,1].set_title('Score Correlation Heatmap')

plt.tight_layout()
plt.savefig('student_eda.png', dpi=150)
plt.show()

## 3. Feature Engineering

In [ ]:
df = data.copy()
le = LabelEncoder()
for col in ['gender','parental_education','lunch','test_prep']:
    df[col] = le.fit_transform(df[col])

X = df[['gender','parental_education','lunch','test_prep',
         'math_score','reading_score','writing_score']]
print('Feature matrix shape:', X.shape)

## 4. Linear Regression – Predict Avg Score

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, df['avg_score'], test_size=0.2, random_state=42)
lr = LinearRegression()
lr.fit(X_tr, y_tr)
mae = mean_absolute_error(y_te, lr.predict(X_te))
print(f'MAE (Linear Regression): {mae:.2f}')

## 5. Decision Tree – Classify Grade

In [ ]:
y_c = df['grade'].astype(str)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, y_c, test_size=0.2, random_state=42)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(Xc_tr, yc_tr)
acc = accuracy_score(yc_te, dt.predict(Xc_te))
print(f'Accuracy (Decision Tree): {acc*100:.1f}%')
print()
print(classification_report(yc_te, dt.predict(Xc_te)))

## 6. Key Insights

In [ ]:
prep_diff  = data.groupby('test_prep')['avg_score'].mean().diff().iloc[-1]
lunch_diff = data.groupby('lunch')['avg_score'].mean().diff().iloc[-1]
best_edu   = data.groupby('parental_education')['avg_score'].mean().idxmax()

print(f'• Test prep students scored {prep_diff:+.1f} pts higher on average')
print(f'• Best parental education group : {best_edu}')
print(f'• Standard lunch outperforms free/reduced by {lunch_diff:+.1f} pts')